In [1]:
import numpy as np
from collections import Counter

In [2]:
class Node:
    def __init__(self, feature=None, threshold=None, left=None, right=None, value=None):
        self.feature = feature
        self.threshold = threshold
        self.left = left
        self.right = right
        self.value = value


def entropy(y):
    classes = np.unique(y)
    ent = 0
    for c in classes:
        p = np.sum(y == c) / len(y)
        if p > 0:
            ent -= p * np.log2(p)
    return ent


def best_split(X, y):
    best_gain = -1
    split_f, split_t = None, None

    for f in range(X.shape[1]):
        for t in np.unique(X[:, f]):

            left = y[X[:, f] <= t]
            right = y[X[:, f] > t]

            if len(left) == 0 or len(right) == 0:
                continue

            parent = entropy(y)
            child = (len(left)/len(y))*entropy(left) + (len(right)/len(y))*entropy(right)
            gain = parent - child

            if gain > best_gain:
                best_gain = gain
                split_f = f
                split_t = t

    return split_f, split_t


def build_tree(X, y, depth=0, max_depth=3):
    if len(np.unique(y)) == 1 or depth == max_depth:
        return Node(value=Counter(y).most_common(1)[0][0])

    f, t = best_split(X, y)

    if f is None:
        return Node(value=Counter(y).most_common(1)[0][0])

    left_mask = X[:, f] <= t
    right_mask = X[:, f] > t

    left = build_tree(X[left_mask], y[left_mask], depth+1, max_depth)
    right = build_tree(X[right_mask], y[right_mask], depth+1, max_depth)

    return Node(f, t, left, right)

In [3]:
def predict_one(x, node):
    if node.value is not None:
        return node.value

    if x[node.feature] <= node.threshold:
        return predict_one(x, node.left)
    return predict_one(x, node.right)

In [4]:
class RandomForest:

    def __init__(self, n_trees=5, max_depth=3, sample_size=0.8):
        self.n_trees = n_trees
        self.max_depth = max_depth
        self.sample_size = sample_size
        self.trees = []

    def fit(self, X, y):

        n_samples = X.shape[0]

        for _ in range(self.n_trees):

            # bootstrap sample
            idxs = np.random.choice(n_samples, int(self.sample_size*n_samples), replace=True)

            X_sample = X[idxs]
            y_sample = y[idxs]

            tree = build_tree(X_sample, y_sample, max_depth=self.max_depth)
            self.trees.append(tree)

    def predict(self, X):

        tree_preds = []

        for tree in self.trees:

            preds = [predict_one(x, tree) for x in X]
            tree_preds.append(preds)

        # transpose: (trees → samples)
        tree_preds = np.array(tree_preds).T

        final_preds = []

        for row in tree_preds:
            final_preds.append(Counter(row).most_common(1)[0][0])

        return np.array(final_preds)

In [5]:
X = np.array([
    [22,25000],
    [25,30000],
    [35,60000],
    [40,70000],
    [45,80000]
])
y = np.array([0,0,1,1,1])

rf = RandomForest(n_trees=5, max_depth=3)
rf.fit(X, y)
X_test = np.array([
    [20,20000],
    [38,65000],
    [50,90000]
])

print(rf.predict(X_test))

[0 1 1]
